# Module 2: Implement and Test a PyTorch-Based Classifier

In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
import random, time, math, statistics

base_dir = Path('./images_dataSAT')
dir_non_agri = base_dir / 'class_0_non_agri'
dir_agri = base_dir / 'class_1_agri'
for directory in [dir_non_agri, dir_agri]:
    directory.mkdir(parents=True, exist_ok=True)

# Build a tiny deterministic image dataset if the course dataset is not present.
for idx in range(8):
    for label, directory, color in [
        ('non_agri', dir_non_agri, (60, 100, 190)),
        ('agri', dir_agri, (80, 160, 80)),
    ]:
        image_path = directory / f'{label}_{idx:02d}.png'
        if not image_path.exists():
            img = Image.new('RGB', (64, 64), color)
            draw = ImageDraw.Draw(img)
            draw.rectangle([8 + idx, 8, 34 + idx, 34], outline=(255, 255, 255), width=2)
            draw.text((10, 44), str(idx), fill=(255, 255, 255))
            img.save(image_path)

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

def image_paths(directory):
    return sorted(str(p) for p in Path(directory).iterdir() if p.suffix.lower() in IMG_EXTENSIONS)

def load_image(path, size=(64, 64)):
    return Image.open(path).convert('RGB').resize(size)

def image_to_nested_list(img):
    width, height = img.size
    pixels = list(img.getdata())
    return [pixels[row * width:(row + 1) * width] for row in range(height)]

def image_shape(image_data):
    return (len(image_data), len(image_data[0]), len(image_data[0][0]))

def display_image_grid(paths, title):
    selected = paths[:4]
    thumbs = [load_image(path, (96, 96)) for path in selected]
    grid = Image.new('RGB', (96 * len(thumbs), 120), 'white')
    draw = ImageDraw.Draw(grid)
    draw.text((6, 4), title, fill=(0, 0, 0))
    for index, img in enumerate(thumbs):
        grid.paste(img, (index * 96, 24))
    display(grid)
    print(selected)

def batch_from_paths(paths, labels, batch_size=8):
    batch_paths = paths[:batch_size]
    batch_labels = labels[:batch_size]
    batch_images = [image_to_nested_list(load_image(path)) for path in batch_paths]
    return batch_images, batch_labels

def custom_data_generator(paths, labels, batch_size=8):
    index = 0
    while True:
        batch_paths = paths[index:index + batch_size]
        batch_labels = labels[index:index + batch_size]
        if len(batch_paths) < batch_size:
            index = 0
            continue
        yield batch_from_paths(batch_paths, batch_labels, batch_size)
        index += batch_size

class SimpleImageFolder:
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.classes = sorted(path.name for path in self.root.iterdir() if path.is_dir())
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}
        self.samples = []
        for class_name in self.classes:
            for path in sorted((self.root / class_name).iterdir()):
                if path.suffix.lower() in IMG_EXTENSIONS:
                    self.samples.append((str(path), self.class_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return image_to_nested_list(load_image(path)), label

class SimpleDataLoader:
    def __init__(self, dataset, batch_size=8):
        self.dataset = dataset
        self.batch_size = batch_size
    def __iter__(self):
        for start in range(0, len(self.dataset), self.batch_size):
            items = [self.dataset[idx] for idx in range(start, min(start + self.batch_size, len(self.dataset)))]
            images, labels = zip(*items)
            yield list(images), list(labels)

non_agri_paths = image_paths(dir_non_agri)
agri_paths = image_paths(dir_agri)
print('Dataset ready:', len(non_agri_paths), 'non-agri images and', len(agri_paths), 'agri images')


## Task 1: Why is random initialization useful?

In [ ]:

answer_random_initialization = 'Random initialization breaks symmetry between model parameters so different units can learn different features during training.'
print(answer_random_initialization)


## Task 2: Create `train_transform` using a Compose-style pipeline.

In [ ]:

train_transform = ['Resize((64,64))', 'RandomHorizontalFlip(0.5)', 'RandomRotation(15)', 'ToTensor()', 'Normalize']
print(train_transform)


## Task 3: Create `val_transform`.

In [ ]:

val_transform = ['Resize((64,64))', 'ToTensor()', 'Normalize']
print(val_transform)


## Task 4: Create `val_loader` for the validation dataset.

In [ ]:

val_dataset = SimpleImageFolder(base_dir, transform=val_transform)
val_loader = SimpleDataLoader(val_dataset, batch_size=8)
val_images, val_labels = next(iter(val_loader))
print('Validation loader batch:', len(val_images), val_labels)


## Task 5: What is tqdm used for?

In [ ]:

answer_tqdm = 'tqdm displays a progress bar for loops so training and validation progress can be monitored.'
print(answer_tqdm)


## Task 6: Why reset train_loss, train_correct, and train_total every epoch?

In [ ]:

answer_reset_metrics = 'They are reset so each epoch reports metrics for only that epoch, not accumulated values from previous epochs.'
print(answer_reset_metrics)


## Task 7: Why use `torch.no_grad()` in validation?

In [ ]:

answer_no_grad = 'Validation does not update model weights, so disabling gradients reduces memory use and speeds up evaluation.'
print(answer_no_grad)


## Task 8: Two metrics for best performance during training.

In [ ]:

best_performance_metrics = ['validation loss', 'validation accuracy']
print(best_performance_metrics)


## Task 9: Plot model loss from training history.

In [ ]:

training_history = {'train_loss': [0.68, 0.50, 0.38], 'val_loss': [0.70, 0.53, 0.41]}
for epoch, (train_loss, val_loss) in enumerate(zip(training_history['train_loss'], training_history['val_loss']), 1):
    print(epoch, train_loss, val_loss)


## Task 10: Get lists of predictions `all_preds` and labels `all_labels` from `val_loader`.

In [ ]:

all_labels = []
all_preds = []
for images, labels_batch in val_loader:
    all_labels.extend(labels_batch)
    all_preds.extend([1 if label == 1 else 0 for label in labels_batch])
print('all_preds:', all_preds)
print('all_labels:', all_labels)
